##Load the Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Installation


In [ ]:
!pip install langchain sentence-transformers chromadb llama-cpp-python langchain_community pypdf



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
pip install langchain_community


In [ ]:
pip install pypdf

In [ ]:
pip install chromadb

##Importing Libraries



In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA, LLMChain



##Import the Document

In [ ]:
loader = PyPDFDirectoryLoader("./RAG")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
docs[1]

##Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
chunks[300]

##Embeddings creations

In [ ]:
import os
from getpass import getpass

os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Enter your Hugging Face Token: ")

In [ ]:
 embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

##Vector Store Creation

In [ ]:

vectorstore = Chroma.from_documents(chunks, embeddings)

In [ ]:
query = "Course Outcomes of Discrete Mathematics"
search_results = vectorstore.similarity_search(query)

In [ ]:
search_results

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k':5})

In [ ]:
retriever.get_relevant_documents(query)

##Hugging face set up

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="TheBloke/CapybaraHermes-2.5-Mistral-7B-GGUF",
    local_dir="./LLMMODEL"
)

In [ ]:
!ls -1 /content/drive/MyDrive/LLMMODEL

In [ ]:
!ls /content/drive/MyDrive/


In [ ]:
!mkdir -p "/content/drive/MyDrive/LLMMODEL"


In [ ]:
!ls "/content/drive/MyDrive/LLMMODEL"


In [ ]:
model_path = "./LLMMODEL"

In [ ]:
!chmod 777 ./LLMMODEL

In [ ]:
!pip install llama-cpp-python

##LLM Model Loading

In [ ]:
from langchain_community.llms import LlamaCpp  # Change Llamacpp to LlamaCpp
llm = LlamaCpp(
    model_path="./LLMMODEL/capybarahermes-2.5-mistral-7b.Q2_K.gguf",
    temperature=0.2,
    max_tokens=2048,
    top_p=1
)

##Use LLM and retriver and query, to generate final response

In [ ]:
template = """
<|context|>
You are a Educational Assistant that follows the instructions and generate the accurate response based on the query and the context provided
Please be truthful and given direct answers.
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke(query)

In [ ]:
response

In [ ]:
import sys

while True:
  user_input = input(f"Input query: ")
  if user_input == "exit":
    print("Exiting...")
    break
  if user_input =="":
    continue
  result = rag_chain.invoke(user_input)
  print("Chatter Mind: ",result)